
# Checkpoint notebook: hybrid-system identification with physics-informed and DHAL-style models

This notebook is a compact, standalone checkpoint of the full exploration so far.

It keeps the core ideas and math needed for a new reader to understand and reproduce the pipeline, while avoiding the proliferation of many separate notebook variants.

## What this notebook contains

We organize everything into two tracks.

### Track A: physics-informed hybrid model
This track assumes the local node dynamics family is known.

- **A-bank**: a physics-informed mode bank
- **A-DHAL**: a DHAL-style hard discrete selector on top of that bank

### Track B: reduced-structure hybrid model
This track relaxes the exact physics assumption and only keeps graph-coupled hybrid structure.

- **B-bank**: a reduced-structure mode bank
- **B-DHAL**: a DHAL-style hard discrete selector on top of that bank

## Design choice in this checkpoint

Earlier notebooks separated:
- A1 then A2,
- B1 then B2.

Here we **compress** that into:
- **A-track = bank + DHAL selector**
- **B-track = bank + DHAL selector**

We still keep the same logic:
- first train the bank,
- then train the selector,
- and optionally do a short joint fine-tuning pass.

This preserves stability, but keeps the notebook conceptually cleaner.

## Why this checkpoint matters

The experiments so far suggest:
- A-side works well and gives a useful reference.
- B-side is harder because the latent bank is not strongly identifiable.
- On the current synthetic dataset, prediction can be good even when the discovered latent modes are not the true modes.

So this notebook should be treated as a **reproducible checkpoint**, not the final word on unsupervised hybrid discovery.



## 0. Imports and runtime setup

### Goal
Load dependencies and set plotting defaults.

### Output to inspect
The selected runtime device.


In [ ]:

import itertools
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



## 1. True hybrid simulator

### Goal
Define the ground-truth switched hybrid system used for synthetic data generation.

### Mathematical formulation
There are $R=3$ modes.  
Each mode $r$ contains:
- graph $W^{(r)} \in \mathbb{R}^{N\times N}$,
- gains $(\alpha^{(r)}, \beta^{(r)})$,
- node-wise Sprott parameter vector $\mu^{(r)} \in \mathbb{R}^N$.

For node state $x_i = [x_i, y_i, z_i]^\top$, the local Sprott dynamics are
$$
f_{\mu_i}(x_i) =
\begin{bmatrix}
-\mu_i x_i - z_i + y_i^2 \\
-x_i - \mu_i y_i + z_i^2 \\
-y_i - \mu_i z_i + x_i^2
\end{bmatrix}.
$$

Stacking all nodes into $X \in \mathbb{R}^{N\times 3}$, mode $r$ evolves as
$$
\dot X =
F_{\mu^{(r)}}^{\mathrm{Sprott}}(X)
+
\alpha^{(r)} D(W^{(r)})X
+
\beta^{(r)} W^{(r)}X,
$$
where $D(W)$ is the degree matrix.

A piecewise-constant mode sequence $z_k \in \{0,1,2\}$ selects which mode is active at each time step.


In [ ]:

A0 = np.array([[0,1,1,1],[1,0,1,0],[1,1,0,1],[1,0,1,0]], dtype=float)
A1 = np.array([[0,1,0,1],[1,0,1,0],[0,1,0,1],[1,0,1,0]], dtype=float)
A2 = np.array([[0,1,1,1],[1,0,1,1],[1,1,0,1],[1,1,1,0]], dtype=float)

true_modes = [
    {"name":"paper_graph_stable","A":A0,"alpha":0.0115,"beta":-0.0115,"mu_vec":np.array([0.55,0.55,0.55,0.55],dtype=float)},
    {"name":"cycle4_heterogeneous_nearcritical","A":A1,"alpha":0.0160,"beta":-0.0120,"mu_vec":np.array([0.55,0.45,0.55,0.45],dtype=float)},
    {"name":"complete4_mixed","A":A2,"alpha":0.0200,"beta":-0.0100,"mu_vec":np.array([0.35,0.55,0.35,0.55],dtype=float)},
]

for k, mode in enumerate(true_modes):
    print(f"Mode {k}: {mode['name']}")
    print("A =")
    print(mode["A"].astype(int))
    print("alpha =", mode["alpha"], "beta =", mode["beta"])
    print("mu_vec =", mode["mu_vec"])
    print()

def sprott_node_mu(state, mu):
    x, y, z = state
    return np.array([-mu*x - z + y**2, -x - mu*y + z**2, -y - mu*z + x**2], dtype=float)

def network_rhs_hybrid(t, X, mode):
    W = mode["A"]; alpha = mode["alpha"]; beta = mode["beta"]; mu_vec = mode["mu_vec"]
    N = W.shape[0]
    D = np.diag(W.sum(axis=1))
    Xmat = X.reshape(N, 3)
    dX = np.zeros_like(Xmat)
    for i in range(N):
        dX[i] = sprott_node_mu(Xmat[i], float(mu_vec[i]))
    dX += alpha * (D @ Xmat) + beta * (W @ Xmat)
    return dX.ravel()

def random_switch_sequence(num_steps, num_modes, min_dwell=6, max_dwell=18, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    modes = []
    current = int(rng.integers(0, num_modes))
    while len(modes) < num_steps:
        dwell = int(rng.integers(min_dwell, max_dwell + 1))
        modes.extend([current] * dwell)
        choices = [m for m in range(num_modes) if m != current]
        current = int(rng.choice(choices))
    return np.array(modes[:num_steps], dtype=int)

def simulate_hybrid_switched_trajectory(x0, mode_seq, dt, mode_bank):
    x = x0.copy().ravel()
    states = [x.reshape(4, 3).copy()]
    for z in mode_seq:
        mode = mode_bank[int(z)]
        sol = solve_ivp(
            lambda t, y: network_rhs_hybrid(t, y, mode),
            (0.0, dt), x, t_eval=[dt],
            rtol=1e-8, atol=1e-10, max_step=dt/5.0, method="RK45"
        )
        x = sol.y[:, -1]
        states.append(x.reshape(4, 3).copy())
    return np.stack(states, axis=0)

dt = 0.05

# Demo trajectory
rng = np.random.default_rng(7)
z_demo = random_switch_sequence(180, len(true_modes), min_dwell=10, max_dwell=22, rng=rng)
x0_demo = rng.uniform(-0.7, 0.7, size=(4, 3))
traj_demo = simulate_hybrid_switched_trajectory(x0_demo, z_demo, dt, true_modes)

t_demo = np.arange(traj_demo.shape[0]) * dt
fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
for c in range(3):
    for i in range(4):
        axes[c].plot(t_demo, traj_demo[:, i, c], label=f"node {i+1}" if c == 0 else None)
    axes[c].set_ylabel(f"ch {c+1}")
axes[3].step(t_demo[:-1], z_demo, where="post")
axes[3].set_ylabel("mode")
axes[3].set_xlabel("time")
axes[0].legend(ncol=4, fontsize=8)
fig.suptitle("Demo hybrid switched trajectory")
plt.tight_layout()
plt.show()



## 2. Dataset construction

### Goal
Construct one-step prediction pairs and history windows.

### Mathematical formulation
Each supervised sample contains:
- current state $X_k$,
- next state $X_{k+1}$,
- true mode label $z_k$,
- history window $X_{k-h:k}$.

We use:
- **physical coordinates** for all dynamics models,
- **normalized histories** only for discrete mode selectors.

### Important note
The current synthetic dataset is approximately balanced across modes.  
This is convenient for training, but it also means that a weak latent selector can sometimes produce nearly balanced mode usage without truly discovering the hybrid modes.


In [ ]:

def build_hybrid_dataset(num_traj=220, steps_per_traj=160, history_len=12, init_scale=0.8, min_dwell=6, max_dwell=18, seed=0):
    rng = np.random.default_rng(seed)
    X_list, Y_list, Z_list, H_list = [], [], [], []
    for _ in range(num_traj):
        x0 = rng.uniform(-init_scale, init_scale, size=(4, 3))
        z = random_switch_sequence(steps_per_traj, len(true_modes), min_dwell=min_dwell, max_dwell=max_dwell, rng=rng)
        traj = simulate_hybrid_switched_trajectory(x0, z, dt, true_modes)
        for k in range(history_len - 1, steps_per_traj):
            H_list.append(traj[k-history_len+1:k+1])
            X_list.append(traj[k])
            Y_list.append(traj[k+1])
            Z_list.append(z[k])
    return np.stack(X_list), np.stack(Y_list), np.array(Z_list), np.stack(H_list)

history_len = 12
X_all, Y_all, Z_all, H_all = build_hybrid_dataset(history_len=history_len)

perm = np.random.default_rng(123).permutation(X_all.shape[0])
n_train = int(0.8 * X_all.shape[0])
train_idx, test_idx = perm[:n_train], perm[n_train:]

X_train, X_test = X_all[train_idx], X_all[test_idx]
Y_train, Y_test = Y_all[train_idx], Y_all[test_idx]
Z_train, Z_test = Z_all[train_idx], Z_all[test_idx]
H_train, H_test = H_all[train_idx], H_all[test_idx]

x_mean = X_train.mean(axis=(0, 1))
x_std = X_train.std(axis=(0, 1)) + 1e-6

def norm_hist(H):
    return (H - x_mean[None, None, None, :]) / x_std[None, None, None, :]

H_train_n = norm_hist(H_train)
H_test_n = norm_hist(H_test)

sup_train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                             torch.tensor(Y_train, dtype=torch.float32),
                             torch.tensor(Z_train, dtype=torch.long))
sup_test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                            torch.tensor(Y_test, dtype=torch.float32),
                            torch.tensor(Z_test, dtype=torch.long))

lat_train_ds = TensorDataset(torch.tensor(H_train_n, dtype=torch.float32),
                             torch.tensor(X_train, dtype=torch.float32),
                             torch.tensor(Y_train, dtype=torch.float32),
                             torch.tensor(Z_train, dtype=torch.long))
lat_test_ds = TensorDataset(torch.tensor(H_test_n, dtype=torch.float32),
                            torch.tensor(X_test, dtype=torch.float32),
                            torch.tensor(Y_test, dtype=torch.float32),
                            torch.tensor(Z_test, dtype=torch.long))

sup_train_loader = DataLoader(sup_train_ds, batch_size=256, shuffle=True)
sup_test_loader = DataLoader(sup_test_ds, batch_size=512, shuffle=False)
lat_train_loader = DataLoader(lat_train_ds, batch_size=256, shuffle=True)
lat_test_loader = DataLoader(lat_test_ds, batch_size=512, shuffle=False)

print("Train size:", len(sup_train_ds), "Test size:", len(sup_test_ds))
print("Train mode counts:", np.bincount(Z_train, minlength=len(true_modes)))



## 3. Model classes

### Goal
Define the two mode-bank families and the DHAL-style selector.

We use the same selector architecture for both A-track and B-track, so that the main difference is the bank model itself.

### DHAL-style selector
Given a history window, the selector produces logits
$$
\ell_k = f_{\mathrm{enc}}(X_{k-h:k}),
$$
then a hard one-hot mode assignment is sampled with straight-through Gumbel-softmax:
$$
z_k^{\mathrm{hard}} = \mathrm{STGumbelSoftmax}(\ell_k).
$$

This makes training better aligned with hard discrete switching than a purely soft selector with argmax at test time.


In [ ]:

true_graphs = np.stack([m["A"] for m in true_modes], axis=0)
true_mu = np.stack([m["mu_vec"] for m in true_modes], axis=0)
true_alpha = np.array([m["alpha"] for m in true_modes], dtype=float)
true_beta = np.array([m["beta"] for m in true_modes], dtype=float)

true_graphs_t = torch.tensor(true_graphs, dtype=torch.float32, device=device)
true_mu_t = torch.tensor(true_mu, dtype=torch.float32, device=device)
true_alpha_t = torch.tensor(true_alpha, dtype=torch.float32, device=device)
true_beta_t = torch.tensor(true_beta, dtype=torch.float32, device=device)

def sprott_batch_with_mu(X, mu_vec):
    if mu_vec.dim() == 1:
        mu_vec = mu_vec.unsqueeze(0).expand(X.shape[0], -1)
    x = X[..., 0]; y = X[..., 1]; z = X[..., 2]
    dx = -mu_vec * x - z + y**2
    dy = -x - mu_vec * y + z**2
    dz = -y - mu_vec * z + x**2
    return torch.stack([dx, dy, dz], dim=-1)

class PhysicsHybridCore(nn.Module):
    def forward(self, X, W, mu_vec, alpha, beta, dt):
        B, N, _ = X.shape
        if W.dim() == 2:
            W = W.unsqueeze(0).expand(B, -1, -1)
        if mu_vec.dim() == 1:
            mu_vec = mu_vec.unsqueeze(0).expand(B, -1)
        if alpha.dim() == 0:
            alpha = alpha.unsqueeze(0).expand(B)
        if beta.dim() == 0:
            beta = beta.unsqueeze(0).expand(B)
        local = sprott_batch_with_mu(X, mu_vec)
        deg = W.sum(dim=-1)
        coupling = alpha[:, None, None] * deg.unsqueeze(-1) * X + beta[:, None, None] * torch.einsum("bij,bjd->bid", W, X)
        return X + dt * (local + coupling)

class LearnableModeBank(nn.Module):
    def __init__(self, num_modes=3, num_nodes=4, mu_min=0.2, mu_max=0.7):
        super().__init__()
        self.mu_min = mu_min; self.mu_max = mu_max
        self.graph_logits = nn.Parameter(torch.zeros(num_modes, num_nodes, num_nodes))
        self.mu_logits = nn.Parameter(torch.zeros(num_modes, num_nodes))
        self.alpha_param = nn.Parameter(torch.zeros(num_modes))
        self.beta_param = nn.Parameter(torch.zeros(num_modes))

        A_init = np.clip(true_graphs, 1e-3, 1 - 1e-3)
        A_logits = np.log(A_init / (1 - A_init))
        mu_scaled = (true_mu - mu_min) / (mu_max - mu_min)
        mu_scaled = np.clip(mu_scaled, 1e-3, 1 - 1e-3)
        mu_logits = np.log(mu_scaled / (1 - mu_scaled))
        with torch.no_grad():
            self.graph_logits.copy_(torch.tensor(A_logits, dtype=torch.float32))
            self.mu_logits.copy_(torch.tensor(mu_logits, dtype=torch.float32))
            self.alpha_param.copy_(torch.tensor(true_alpha, dtype=torch.float32))
            self.beta_param.copy_(torch.tensor(true_beta, dtype=torch.float32))

    def graphs(self):
        S = 0.5 * (self.graph_logits + self.graph_logits.transpose(1, 2))
        W = torch.sigmoid(S)
        eye = torch.eye(W.shape[-1], device=W.device).unsqueeze(0)
        return W * (1.0 - eye)

    def mus(self):
        mu01 = torch.sigmoid(self.mu_logits)
        return self.mu_min + (self.mu_max - self.mu_min) * mu01

    def alphas(self): return self.alpha_param
    def betas(self): return self.beta_param

class PhysicsInformedBank(nn.Module):
    def __init__(self, num_modes=3):
        super().__init__()
        self.mode_bank = LearnableModeBank(num_modes=num_modes)
        self.core = PhysicsHybridCore()

    def mode_step(self, X, mode_idx, dt):
        Wb = self.mode_bank.graphs()
        mub = self.mode_bank.mus()
        ab = self.mode_bank.alphas()
        bb = self.mode_bank.betas()
        return self.core(X, Wb[mode_idx], mub[mode_idx], ab[mode_idx], bb[mode_idx], dt)

    def forward_grouped(self, X, z, dt):
        out = torch.zeros_like(X)
        for mode_idx in range(self.mode_bank.graphs().shape[0]):
            idx = torch.nonzero(z == mode_idx, as_tuple=False).squeeze(-1)
            if idx.numel() == 0:
                continue
            out[idx] = self.mode_step(X[idx], mode_idx, dt)
        return out

class ReducedStructureModeBank(nn.Module):
    def __init__(self, num_modes=3, num_nodes=4):
        super().__init__()
        self.graph_logits = nn.Parameter(torch.zeros(num_modes, num_nodes, num_nodes))
        self.alpha_param = nn.Parameter(torch.zeros(num_modes))
        self.beta_param = nn.Parameter(torch.zeros(num_modes))

    def graphs(self):
        S = 0.5 * (self.graph_logits + self.graph_logits.transpose(1, 2))
        W = torch.sigmoid(S)
        eye = torch.eye(W.shape[-1], device=W.device).unsqueeze(0)
        return W * (1.0 - eye)

    def alphas(self): return self.alpha_param
    def betas(self): return self.beta_param

class SmallMLP(nn.Module):
    def __init__(self, in_dim=3, hidden_dim=8, out_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

class ReducedStructureBank(nn.Module):
    def __init__(self, num_modes=3, shared_hidden=16, residual_hidden=8):
        super().__init__()
        self.num_modes = num_modes
        self.bank = ReducedStructureModeBank(num_modes=num_modes)
        self.shared_field = nn.Sequential(
            nn.Linear(3, shared_hidden), nn.Tanh(),
            nn.Linear(shared_hidden, 3),
        )
        self.residual_fields = nn.ModuleList([SmallMLP(3, residual_hidden, 3) for _ in range(num_modes)])

    def mode_step(self, X, mode_idx, dt):
        W = self.bank.graphs()[mode_idx]
        a = self.bank.alphas()[mode_idx]
        b = self.bank.betas()[mode_idx]
        B, N, _ = X.shape
        xflat = X.reshape(B * N, 3)
        local = self.shared_field(xflat) + self.residual_fields[mode_idx](xflat)
        local = local.reshape(B, N, 3)
        deg = W.sum(dim=-1)
        coupling = a * deg.unsqueeze(0).unsqueeze(-1) * X + b * torch.einsum("ij,bjd->bid", W, X)
        return X + dt * (local + coupling)

    def forward_grouped(self, X, z, dt):
        out = torch.zeros_like(X)
        for mode_idx in range(self.num_modes):
            idx = torch.nonzero(z == mode_idx, as_tuple=False).squeeze(-1)
            if idx.numel() == 0:
                continue
            out[idx] = self.mode_step(X[idx], mode_idx, dt)
        return out

class GRUSelector(nn.Module):
    def __init__(self, history_len=12, num_nodes=4, state_dim=3, hidden_dim=64, num_modes=3):
        super().__init__()
        self.input_dim = num_nodes * state_dim
        self.gru = nn.GRU(input_size=self.input_dim, hidden_size=hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, num_modes)
    def forward(self, H):
        B, T, N, d = H.shape
        x = H.reshape(B, T, N * d)
        out, h = self.gru(x)
        return self.head(h[-1])

def st_gumbel_onehot(logits, tau=1.0):
    return torch.nn.functional.gumbel_softmax(logits, tau=tau, hard=True, dim=-1)

def hard_indices(onehot):
    return onehot.argmax(dim=-1)

class HybridSelectorModel(nn.Module):
    def __init__(self, bank_model, history_len=12, num_modes=3):
        super().__init__()
        self.selector = GRUSelector(history_len=history_len, num_modes=num_modes)
        self.bank_model = bank_model

    def forward(self, H_norm, X_phys, dt, tau=1.0):
        logits = self.selector(H_norm)
        onehot = st_gumbel_onehot(logits, tau=tau)
        idx = hard_indices(onehot)
        pred = self.bank_model.forward_grouped(X_phys, idx, dt)
        return pred, logits, onehot, idx



## 4. Loss functions

### Track A: physics-informed bank pretraining
The physics-informed bank is trained with:
$$
\mathcal L_{A\text{-bank}}
=
\mathcal L_{\mathrm{pred}}
+
\mathcal L_W
+
\mathcal L_{\mu}
+
\mathcal L_g
+
10^{-3}\mathcal L_{\mathrm{bin}}.
$$

### Track A: DHAL selector
The A-track selector is trained with:
$$
\mathcal L_{A\text{-DHAL}}
=
\mathcal L_{\mathrm{pred}}
+
0.1\mathcal L_{\mathrm{cls}}
+
10^{-2}\mathcal L_{\mathrm{bal}}
+
10^{-3}\mathcal L_{\mathrm{dwell}}.
$$

### Track B: reduced-structure bank pretraining
The reduced-structure bank is trained with:
$$
\mathcal L_{B\text{-bank}}
=
\mathcal L_{\mathrm{pred}}
+
10^{-3}\mathcal L_{\mathrm{bin}}
+
10^{-4}\mathcal L_{\mathrm{sparse}}
+
10^{-3}\mathcal L_{\mathrm{sep}}.
$$

### Track B: DHAL selector
The B-track selector is trained without labels:
$$
\mathcal L_{B\text{-DHAL}}
=
\mathcal L_{\mathrm{pred}}
+
10^{-2}\mathcal L_{\mathrm{bal}}
+
10^{-3}\mathcal L_{\mathrm{dwell}}.
$$

### Definitions
Prediction loss:
$$
\mathcal L_{\mathrm{pred}}
=
\frac{1}{M}\sum_k \|\widehat X_{k+1} - X_{k+1}\|_F^2
$$

Supervised selector classification loss:
$$
\mathcal L_{\mathrm{cls}}
=
\frac{1}{M}\sum_k \mathrm{CE}(\ell_k, z_k)
$$

Hard assignment balance loss:
$$
\mathcal L_{\mathrm{bal}}
=
\left\|
\frac{1}{M}\sum_k z_k^{\mathrm{hard}}
-
\frac{1}{R}\mathbf 1
\right\|_2^2
$$

Dwell loss:
$$
\mathcal L_{\mathrm{dwell}}
=
\frac{1}{M-1}\sum_k
\left\|
z_{k+1}^{\mathrm{hard}} - z_k^{\mathrm{hard}}
\right\|_1
$$

Binary graph regularizer:
$$
\mathcal L_{\mathrm{bin}}
=
\frac{1}{R N^2}\sum_{r,i,j} W_{ij}^{(r)}(1-W_{ij}^{(r)})
$$

Graph sparsity:
$$
\mathcal L_{\mathrm{sparse}}
=
\frac{1}{R}\sum_r \sum_{i<j} W_{ij}^{(r)}
$$

Graph separation:
$$
\mathcal L_{\mathrm{sep}}
=
-\frac{1}{R(R-1)}
\sum_{r\neq s}
\|W_r - W_s\|_F^2.
$$



## 5. Training A-track and B-track

### Goal
Train:
- A-bank then A-DHAL
- B-bank then B-DHAL

### Why keep the staged structure here?
Although one could try direct end-to-end DHAL training from scratch, staged bank training is more stable and easier to interpret. This checkpoint keeps that structure, but conceptually compresses everything into the two tracks.


In [ ]:

def eval_bank(model, loader):
    model.eval()
    total = 0.0; count = 0
    with torch.no_grad():
        for xb, yb, zb in loader:
            xb = xb.to(device); yb = yb.to(device); zb = zb.to(device)
            pred = model.forward_grouped(xb, zb, dt)
            total += ((pred - yb) ** 2).mean().item() * xb.shape[0]
            count += xb.shape[0]
    return total / count

def eval_selector(model, loader, supervised=True, tau=0.35):
    model.eval()
    total = 0.0; count = 0; correct = 0
    with torch.no_grad():
        for hb, xb, yb, zb in loader:
            hb = hb.to(device); xb = xb.to(device); yb = yb.to(device); zb = zb.to(device)
            pred, logits, onehot, idx = model(hb, xb, dt, tau=tau)
            total += ((pred - yb) ** 2).mean().item() * xb.shape[0]
            count += xb.shape[0]
            if supervised:
                correct += (idx == zb).sum().item()
    if supervised:
        return total / count, correct / count
    return total / count

# ---- A-bank pretraining ----
torch.manual_seed(0)
A_bank = PhysicsInformedBank(num_modes=len(true_modes)).to(device)
opt_A_bank = optim.Adam(A_bank.parameters(), lr=2e-3, weight_decay=1e-6)

A_bank_train, A_bank_test = [], []
for epoch in range(1, 41):
    A_bank.train()
    running = 0.0; seen = 0
    for xb, yb, zb in sup_train_loader:
        xb = xb.to(device); yb = yb.to(device); zb = zb.to(device)
        pred = A_bank.forward_grouped(xb, zb, dt)
        pred_loss = ((pred - yb) ** 2).mean()

        W = A_bank.mode_bank.graphs()
        mu = A_bank.mode_bank.mus()
        a = A_bank.mode_bank.alphas()
        b = A_bank.mode_bank.betas()

        loss = pred_loss + ((W - true_graphs_t) ** 2).mean() + ((mu - true_mu_t) ** 2).mean()                + ((a - true_alpha_t) ** 2).mean() + ((b - true_beta_t) ** 2).mean()                + 1e-3 * (W * (1 - W)).mean()

        opt_A_bank.zero_grad(); loss.backward(); opt_A_bank.step()
        running += pred_loss.item() * xb.shape[0]; seen += xb.shape[0]

    A_bank_train.append(running / seen)
    A_bank_test.append(eval_bank(A_bank, sup_test_loader))

# ---- A-DHAL selector ----
A_track = HybridSelectorModel(A_bank, history_len=history_len, num_modes=len(true_modes)).to(device)
for p in A_track.bank_model.parameters():
    p.requires_grad = False
opt_A_sel = optim.Adam(A_track.selector.parameters(), lr=2e-3, weight_decay=1e-6)

A_sel_train, A_sel_test, A_sel_acc = [], [], []
for epoch in range(1, 81):
    tau = max(0.4, 1.2 * (0.985 ** epoch))
    A_track.train()
    running = 0.0; seen = 0
    for hb, xb, yb, zb in lat_train_loader:
        hb = hb.to(device); xb = xb.to(device); yb = yb.to(device); zb = zb.to(device)
        pred, logits, onehot, idx = A_track(hb, xb, dt, tau=tau)
        pred_loss = ((pred - yb) ** 2).mean()
        cls_loss = nn.CrossEntropyLoss()(logits, zb)
        balance = ((onehot.float().mean(dim=0) - 1.0 / len(true_modes)) ** 2).mean()
        dwell = torch.abs(onehot[1:].float() - onehot[:-1].float()).mean() if onehot.shape[0] > 1 else 0.0
        loss = pred_loss + 0.1 * cls_loss + 1e-2 * balance + 1e-3 * dwell

        opt_A_sel.zero_grad(); loss.backward(); opt_A_sel.step()
        running += pred_loss.item() * xb.shape[0]; seen += xb.shape[0]

    test_loss, test_acc = eval_selector(A_track, lat_test_loader, supervised=True, tau=tau)
    A_sel_train.append(running / seen)
    A_sel_test.append(test_loss)
    A_sel_acc.append(test_acc)

# ---- optional short A joint fine-tune ----
for p in A_track.bank_model.parameters():
    p.requires_grad = True
opt_A_joint = optim.Adam(A_track.parameters(), lr=5e-4, weight_decay=1e-6)

for epoch in range(1, 11):
    tau = 0.35
    A_track.train()
    for hb, xb, yb, zb in lat_train_loader:
        hb = hb.to(device); xb = xb.to(device); yb = yb.to(device); zb = zb.to(device)
        pred, logits, onehot, idx = A_track(hb, xb, dt, tau=tau)
        pred_loss = ((pred - yb) ** 2).mean()
        cls_loss = nn.CrossEntropyLoss()(logits, zb)
        balance = ((onehot.float().mean(dim=0) - 1.0 / len(true_modes)) ** 2).mean()
        dwell = torch.abs(onehot[1:].float() - onehot[:-1].float()).mean() if onehot.shape[0] > 1 else 0.0

        W = A_track.bank_model.mode_bank.graphs()
        mu = A_track.bank_model.mode_bank.mus()
        a = A_track.bank_model.mode_bank.alphas()
        b = A_track.bank_model.mode_bank.betas()
        bank_reg = ((W - true_graphs_t) ** 2).mean() + ((mu - true_mu_t) ** 2).mean()                + ((a - true_alpha_t) ** 2).mean() + ((b - true_beta_t) ** 2).mean()                + 1e-3 * (W * (1 - W)).mean()

        loss = pred_loss + 0.1 * cls_loss + 1e-2 * balance + 1e-3 * dwell + 0.5 * bank_reg
        opt_A_joint.zero_grad(); loss.backward(); opt_A_joint.step()

# ---- B-bank pretraining ----
torch.manual_seed(0)
B_bank = ReducedStructureBank(num_modes=len(true_modes), shared_hidden=16, residual_hidden=8).to(device)
opt_B_bank = optim.Adam(B_bank.parameters(), lr=2e-3, weight_decay=1e-6)

B_bank_train, B_bank_test = [], []
for epoch in range(1, 61):
    B_bank.train()
    running = 0.0; seen = 0
    for xb, yb, zb in sup_train_loader:
        xb = xb.to(device); yb = yb.to(device); zb = zb.to(device)
        pred = B_bank.forward_grouped(xb, zb, dt)
        pred_loss = ((pred - yb) ** 2).mean()

        W = B_bank.bank.graphs()
        binary = (W * (1 - W)).mean()
        sparse = (W.sum() / 2.0) / W.shape[0]
        sep = 0.0
        cnt = 0
        for r in range(W.shape[0]):
            for s in range(r + 1, W.shape[0]):
                sep = sep - ((W[r] - W[s]) ** 2).mean()
                cnt += 1
        sep = sep / max(cnt, 1)

        loss = pred_loss + 1e-3 * binary + 1e-4 * sparse + 1e-3 * sep

        opt_B_bank.zero_grad(); loss.backward(); opt_B_bank.step()
        running += pred_loss.item() * xb.shape[0]; seen += xb.shape[0]

    B_bank_train.append(running / seen)
    B_bank_test.append(eval_bank(B_bank, sup_test_loader))

# ---- B-DHAL selector ----
B_track = HybridSelectorModel(B_bank, history_len=history_len, num_modes=len(true_modes)).to(device)
# warm-start from A selector
B_track.selector.load_state_dict(A_track.selector.state_dict())
for p in B_track.bank_model.parameters():
    p.requires_grad = False
opt_B_sel = optim.Adam(B_track.selector.parameters(), lr=2e-3, weight_decay=1e-6)

B_sel_train, B_sel_test, B_sel_usage = [], [], []
for epoch in range(1, 121):
    tau = max(0.35, 1.2 * (0.985 ** epoch))
    B_track.train()
    running = 0.0; seen = 0
    for hb, xb, yb, zb in lat_train_loader:
        hb = hb.to(device); xb = xb.to(device); yb = yb.to(device)
        pred, logits, onehot, idx = B_track(hb, xb, dt, tau=tau)
        pred_loss = ((pred - yb) ** 2).mean()
        balance = ((onehot.float().mean(dim=0) - 1.0 / len(true_modes)) ** 2).mean()
        dwell = torch.abs(onehot[1:].float() - onehot[:-1].float()).mean() if onehot.shape[0] > 1 else 0.0
        loss = pred_loss + 1e-2 * balance + 1e-3 * dwell

        opt_B_sel.zero_grad(); loss.backward(); opt_B_sel.step()
        running += pred_loss.item() * xb.shape[0]; seen += xb.shape[0]

    B_track.eval()
    total = 0.0; count = 0; usage = np.zeros(len(true_modes), dtype=float)
    with torch.no_grad():
        for hb, xb, yb, zb in lat_test_loader:
            hb = hb.to(device); xb = xb.to(device); yb = yb.to(device)
            pred, logits, onehot, idx = B_track(hb, xb, dt, tau=tau)
            total += ((pred - yb) ** 2).mean().item() * xb.shape[0]
            count += xb.shape[0]
            usage += onehot.float().mean(dim=0).cpu().numpy()

    B_sel_train.append(running / seen)
    B_sel_test.append(total / count)
    B_sel_usage.append(usage / len(lat_test_loader))

# ---- optional short B joint fine-tune ----
for p in B_track.bank_model.parameters():
    p.requires_grad = True
opt_B_joint = optim.Adam(B_track.parameters(), lr=5e-4, weight_decay=1e-6)

for epoch in range(1, 11):
    tau = 0.35
    B_track.train()
    for hb, xb, yb, zb in lat_train_loader:
        hb = hb.to(device); xb = xb.to(device); yb = yb.to(device)
        pred, logits, onehot, idx = B_track(hb, xb, dt, tau=tau)
        pred_loss = ((pred - yb) ** 2).mean()
        balance = ((onehot.float().mean(dim=0) - 1.0 / len(true_modes)) ** 2).mean()
        dwell = torch.abs(onehot[1:].float() - onehot[:-1].float()).mean() if onehot.shape[0] > 1 else 0.0

        W = B_track.bank_model.bank.graphs()
        binary = (W * (1 - W)).mean()
        sparse = (W.sum() / 2.0) / W.shape[0]
        sep = 0.0
        cnt = 0
        for r in range(W.shape[0]):
            for s in range(r + 1, W.shape[0]):
                sep = sep - ((W[r] - W[s]) ** 2).mean()
                cnt += 1
        sep = sep / max(cnt, 1)

        loss = pred_loss + 1e-2 * balance + 1e-3 * dwell + 0.5 * (1e-3 * binary + 1e-4 * sparse + 1e-3 * sep)
        opt_B_joint.zero_grad(); loss.backward(); opt_B_joint.step()



## 6. Training curves

### Goal
Visualize training for:
- A-bank
- A-DHAL
- B-bank
- B-DHAL


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0,0].plot(A_bank_train, label="train")
axes[0,0].plot(A_bank_test, label="test")
axes[0,0].set_yscale("log")
axes[0,0].set_title("A-bank")
axes[0,0].legend()

axes[0,1].plot(A_sel_train, label="train")
axes[0,1].plot(A_sel_test, label="test")
axes[0,1].set_yscale("log")
axes[0,1].set_title("A-DHAL")
axes[0,1].legend()

axes[1,0].plot(B_bank_train, label="train")
axes[1,0].plot(B_bank_test, label="test")
axes[1,0].set_yscale("log")
axes[1,0].set_title("B-bank")
axes[1,0].legend()

axes[1,1].plot(B_sel_train, label="train")
axes[1,1].plot(B_sel_test, label="test")
axes[1,1].set_yscale("log")
axes[1,1].set_title("B-DHAL")
axes[1,1].legend()

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].plot(A_sel_acc)
axes[0].set_title("A-DHAL accuracy")
usage_arr = np.array(B_sel_usage)
for k in range(usage_arr.shape[1]):
    axes[1].plot(usage_arr[:, k], label=f"mode {k}")
axes[1].set_title("B-DHAL hard mode usage")
axes[1].legend()
plt.tight_layout()
plt.show()



## 7. Diagnostics: banks, confusion matrices, aligned evaluation

### Goal
Inspect what the models learned.

### Diagnostics included
- learned A-bank and B-bank graphs
- A-DHAL confusion matrix
- raw B-DHAL confusion matrix
- aligned B-DHAL confusion matrix
- aligned B-DHAL accuracy

### Important note
For unsupervised B-DHAL, latent labels are only identifiable up to permutation. So the aligned evaluation is the more meaningful one.


In [ ]:

W_A = A_track.bank_model.mode_bank.graphs().detach().cpu().numpy()
W_B = B_track.bank_model.bank.graphs().detach().cpu().numpy()

fig, axes = plt.subplots(3, 3, figsize=(11, 10))
for k in range(3):
    axes[k,0].imshow(true_graphs[k], vmin=0, vmax=1)
    axes[k,0].set_title(f"True mode {k}")
    axes[k,1].imshow(W_A[k], vmin=0, vmax=1)
    axes[k,1].set_title(f"A-bank mode {k}")
    axes[k,2].imshow(W_B[k], vmin=0, vmax=1)
    axes[k,2].set_title(f"B-bank mode {k}")
plt.tight_layout()
plt.show()

def confusion_matrix_numpy(y_true, y_pred, K):
    C = np.zeros((K, K), dtype=int)
    for a, b in zip(y_true, y_pred):
        C[a, b] += 1
    return C

def best_permutation_from_confusion(C):
    K = C.shape[0]
    best_perm = None
    best_score = -1
    for perm in itertools.permutations(range(K)):
        score = sum(C[i, perm[i]] for i in range(K))
        if score > best_score:
            best_score = score
            best_perm = perm
    return best_perm, best_score

def apply_perm(y_pred, perm):
    mapping = {old:new for old, new in enumerate(perm)}
    return np.array([mapping[y] for y in y_pred], dtype=int)

# A-DHAL confusion
A_track.eval()
yt_A, yp_A = [], []
with torch.no_grad():
    for hb, xb, yb, zb in lat_test_loader:
        hb = hb.to(device); xb = xb.to(device)
        _, _, _, idx = A_track(hb, xb, dt, tau=0.35)
        yt_A.append(zb.numpy()); yp_A.append(idx.cpu().numpy())
yt_A = np.concatenate(yt_A); yp_A = np.concatenate(yp_A)
C_A = confusion_matrix_numpy(yt_A, yp_A, len(true_modes))

# B-DHAL confusion
B_track.eval()
yt_B, yp_B = [], []
with torch.no_grad():
    for hb, xb, yb, zb in lat_test_loader:
        hb = hb.to(device); xb = xb.to(device)
        _, _, _, idx = B_track(hb, xb, dt, tau=0.35)
        yt_B.append(zb.numpy()); yp_B.append(idx.cpu().numpy())
yt_B = np.concatenate(yt_B); yp_B = np.concatenate(yp_B)
C_B_raw = confusion_matrix_numpy(yt_B, yp_B, len(true_modes))

perm_best, score_best = best_permutation_from_confusion(C_B_raw)
yp_B_aligned = apply_perm(yp_B, perm_best)
C_B_aligned = confusion_matrix_numpy(yt_B, yp_B_aligned, len(true_modes))
acc_B_aligned = np.trace(C_B_aligned) / C_B_aligned.sum()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
im0 = axes[0].imshow(C_A)
axes[0].set_title("A-DHAL confusion")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(C_B_raw)
axes[1].set_title("B-DHAL raw")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(C_B_aligned)
axes[2].set_title("B-DHAL aligned")
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

print("A-DHAL confusion matrix:")
print(C_A)
print("B-DHAL raw confusion matrix:")
print(C_B_raw)
print("B-DHAL best permutation:", perm_best)
print("B-DHAL aligned confusion matrix:")
print(C_B_aligned)
print("B-DHAL aligned accuracy:", round(float(acc_B_aligned), 4))
print("B-DHAL final hard mode usage:", np.round(np.array(B_sel_usage)[-1], 4))



## 8. Rollout evaluation

### Goal
Restore rollout evaluation so that the notebook remains useful as a standalone checkpoint.

We compare:
- true system rollout,
- A-track rollout with true modes,
- A-track rollout with inferred modes,
- B-track rollout with true modes,
- B-track rollout with inferred modes.

### Why this matters
For hybrid learning, confusion matrices alone do not tell the full story. A model may predict well even when its latent mode interpretation is poor. Rollout evaluation makes that tradeoff visible.


In [ ]:

def rollout_true_and_modes():
    rng = np.random.default_rng(99)
    z_roll = random_switch_sequence(180, len(true_modes), min_dwell=8, max_dwell=20, rng=rng)
    x0_roll = rng.uniform(-0.7, 0.7, size=(4, 3))
    traj_true = simulate_hybrid_switched_trajectory(x0_roll, z_roll, dt, true_modes)
    return traj_true, z_roll

def rollout_bank_with_modes(bank_model, x0_phys, z_seq):
    traj = [x0_phys.copy()]
    with torch.no_grad():
        xt = torch.tensor(x0_phys[None], dtype=torch.float32, device=device)
        for z in z_seq:
            zt = torch.tensor([z], dtype=torch.long, device=device)
            xt = bank_model.forward_grouped(xt, zt, dt)
            traj.append(xt.cpu().numpy()[0].copy())
    return np.stack(traj, axis=0)

def infer_modes_selector(track_model, traj_true_phys, history_len, perm_map=None, tau=0.35):
    zhat_list = []
    with torch.no_grad():
        for k in range(1, traj_true_phys.shape[0]):
            hist = traj_true_phys[:history_len] if k < history_len else traj_true_phys[k-history_len:k]
            hist_n = (hist - x_mean[None, None, :]) / x_std[None, None, :]
            xk = traj_true_phys[k - 1]
            hb = torch.tensor(hist_n[None], dtype=torch.float32, device=device)
            xb = torch.tensor(xk[None], dtype=torch.float32, device=device)
            _, _, _, idx = track_model(hb, xb, dt, tau=tau)
            z = int(idx.item())
            if perm_map is not None:
                z = perm_map[z]
            zhat_list.append(z)
    return np.array(zhat_list, dtype=int)

traj_true, z_roll = rollout_true_and_modes()
t_roll = np.arange(traj_true.shape[0]) * dt

traj_A_true = rollout_bank_with_modes(A_track.bank_model, traj_true[0], z_roll)
traj_B_true = rollout_bank_with_modes(B_track.bank_model, traj_true[0], z_roll)

zhat_A = infer_modes_selector(A_track, traj_true, history_len, perm_map=None, tau=0.35)
perm_map = {old:new for old, new in enumerate(perm_best)}
zhat_B = infer_modes_selector(B_track, traj_true, history_len, perm_map=perm_map, tau=0.35)

traj_A_inf = rollout_bank_with_modes(A_track.bank_model, traj_true[0], zhat_A)
traj_B_inf = rollout_bank_with_modes(B_track.bank_model, traj_true[0], zhat_B)

T_common = min(len(traj_true), len(traj_A_true), len(traj_B_true), len(traj_A_inf), len(traj_B_inf))
traj_true_use = traj_true[:T_common]
traj_A_true = traj_A_true[:T_common]
traj_B_true = traj_B_true[:T_common]
traj_A_inf = traj_A_inf[:T_common]
traj_B_inf = traj_B_inf[:T_common]
t_roll = t_roll[:T_common]
z_roll_use = z_roll[:T_common-1]
zhat_A = zhat_A[:T_common-1]
zhat_B = zhat_B[:T_common-1]

def rmse_traj(pred, true):
    return np.sqrt(np.mean((pred - true) ** 2, axis=(1,2)))

rmse_A_true = rmse_traj(traj_A_true, traj_true_use)
rmse_A_inf = rmse_traj(traj_A_inf, traj_true_use)
rmse_B_true = rmse_traj(traj_B_true, traj_true_use)
rmse_B_inf = rmse_traj(traj_B_inf, traj_true_use)

def summarize_rmse(name, arr):
    print(name, "mean", round(float(arr.mean()), 6),
          "| final", round(float(arr[-1]), 6),
          "| max", round(float(arr.max()), 6))

summarize_rmse("A true-mode rollout", rmse_A_true)
summarize_rmse("A inferred-mode rollout", rmse_A_inf)
summarize_rmse("B true-mode rollout", rmse_B_true)
summarize_rmse("B inferred-mode rollout", rmse_B_inf)


In [ ]:

fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
for c in range(3):
    axes[c].plot(t_roll, traj_true_use[:, 0, c], linewidth=2, label="true")
    axes[c].plot(t_roll, traj_A_true[:, 0, c], "--", linewidth=1.2, label="A true mode")
    axes[c].plot(t_roll, traj_A_inf[:, 0, c], ":", linewidth=1.5, label="A inferred mode")
    axes[c].plot(t_roll, traj_B_true[:, 0, c], "-.", linewidth=1.2, label="B true mode")
    axes[c].plot(t_roll, traj_B_inf[:, 0, c], linewidth=1.0, alpha=0.9, label="B inferred mode")
    axes[c].set_ylabel(f"node1 ch{c+1}")
axes[3].step(t_roll[:-1], z_roll_use, where="post", label="true mode")
axes[3].set_ylabel("mode")
axes[3].set_xlabel("time")
axes[0].legend(ncol=3, fontsize=8)
fig.suptitle("Rollout comparison")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_roll, rmse_A_true, label="A true mode")
ax.plot(t_roll, rmse_A_inf, label="A inferred mode")
ax.plot(t_roll, rmse_B_true, label="B true mode")
ax.plot(t_roll, rmse_B_inf, label="B inferred mode")
ax.set_title("Rollout RMSE")
ax.legend()
plt.show()


In [ ]:

fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)
axes[0].step(t_roll[:-1], z_roll_use, where="post")
axes[0].set_ylabel("true")
axes[1].step(t_roll[:-1], zhat_A, where="post")
axes[1].set_ylabel("A inferred")
axes[2].step(t_roll[:-1], zhat_B, where="post")
axes[2].set_ylabel("B inferred")
axes[2].set_xlabel("time")
fig.suptitle("Inferred mode sequences on rollout")
plt.tight_layout()
plt.show()



## 9. Final remarks

### What this checkpoint shows
- The **A-track** remains the strong reference and is useful for verifying the selector architecture.
- The **B-track** remains harder: prediction can be decent even when latent mode discovery is weak.
- Alignment-based evaluation is necessary for unsupervised modes, but even after alignment the current synthetic setting still appears weak for true unsupervised hybrid-mode recovery.

### Why the current dataset may be weak for B-track discovery
On this dataset, the model can often get good predictive performance without learning the true hybrid modes. Balanced mode counts and relatively forgiving dynamics may let the latent selector "cheat" by distributing modes in a way that is predictive but not semantically correct.

### Next directions beyond this checkpoint
A more revealing synthetic benchmark would likely need one or more of:
- more distinct mode-specific dynamics,
- stronger dwell structure,
- mode imbalance,
- switching tied to explicit state-dependent guards,
- or auxiliary labels/targets that sharpen latent mode semantics.

This notebook is intended to remain a clean, standalone checkpoint for future iterations.


## 10. Morse graph analysis of 4D bistable network

### Goal
Apply MorseGraph-L4DC analysis to compare ground-truth vs learned dynamics of a 4D hybrid system with state-dependent switching.

### System design
Replace the 12D Sprott system with **1D bistable nodes** ($4 \times 1\text{D} = 4\text{D}$ state space).

**Local dynamics**: $g(x_i, \mu_i) = \mu_i x_i - x_i^3$ (pitchfork bifurcation)

**Switching**: $p_1(x) = x_1 + x_2$, $p_2(x) = x_3 + x_4$ give $2^2 = 4$ modes via sign patterns.

### Pipeline
1. Define `SwitchingSystem` with bistable vector fields
2. Generate training data (state-dependent switching)
3. Train physics-informed bank from data
4. Morse graph analysis on ground-truth system
5. Morse graph analysis on learned system
6. Compare topological structure

In [ ]:
# ============================================================
# 10.1 MorseGraph imports + 4D bistable switching system
# ============================================================
import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), 'MorseGraph-L4DC'))

from MorseGraphL4DC.grids import UniformGrid
from MorseGraphL4DC.dynamics import F_integration
from MorseGraphL4DC.systems import SwitchingSystem
from MorseGraphL4DC.analysis import full_morse_graph_analysis

# ---- 4D bistable network parameters ----
# Local dynamics: g(x_i, mu_i) = mu_i * x_i - x_i^3
# Mode r: dx_i/dt = mu_i^(r)*x_i - x_i^3 + alpha^(r)*deg_i*x_i + beta^(r)*sum_j W_ij^(r)*x_j

c1_bi, c2_bi = 0.0, 0.0  # switching thresholds

bistable_polynomials = [
    lambda x, c=c1_bi: x[0] + x[1] - c,
    lambda x, c=c2_bi: x[2] + x[3] - c,
]

# 4 modes (2^2) using adjacency matrices A0, A1, A2 from Section 1
bistable_mode_params = [
    {"A": A0, "mu": np.array([1.0, 1.0, 1.0, 1.0]), "alpha":  0.05, "beta": -0.05},  # mode 0: p1<0, p2<0
    {"A": A1, "mu": np.array([0.8, 0.6, 0.8, 0.6]), "alpha":  0.08, "beta": -0.04},  # mode 1: p1>0, p2<0
    {"A": A2, "mu": np.array([0.6, 0.8, 0.6, 0.8]), "alpha":  0.06, "beta": -0.06},  # mode 2: p1<0, p2>0
    {"A": A2, "mu": np.array([0.5, 0.5, 0.5, 0.5]), "alpha":  0.10, "beta": -0.03},  # mode 3: p1>0, p2>0
]

def make_bistable_vf(A, mu, alpha, beta):
    """Create a bistable network vector field closure."""
    deg = A.sum(axis=1)
    def vf(x):
        return mu * x - x**3 + alpha * deg * x + beta * A @ x
    return vf

bistable_vfs = [make_bistable_vf(**m) for m in bistable_mode_params]
bistable_system = SwitchingSystem(bistable_polynomials, bistable_vfs)

# ---- Verify: test trajectory ----
x0_test = np.array([0.5, -0.3, 0.4, -0.6])
sol_test = solve_ivp(bistable_system, [0, 10.0], x0_test, max_step=0.05, dense_output=True)
t_vis = np.linspace(0, 10.0, 1000)
x_vis = sol_test.sol(t_vis)

fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
for i in range(4):
    axes[0].plot(t_vis, x_vis[i], label=f'$x_{i+1}$')
axes[0].set_ylabel('state')
axes[0].legend(ncol=4, fontsize=8)
modes_vis = np.array([bistable_system.sigma(x_vis[:, k]) for k in range(len(t_vis))])
axes[1].step(t_vis, modes_vis, where='post')
axes[1].set_ylabel('mode')
axes[1].set_xlabel('time')
fig.suptitle('4D bistable network: test trajectory with state-dependent switching')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 10.2 Data generation from 4D bistable system
# ============================================================
dt_bistable = 0.1   # one-step prediction interval
T_bistable = 20.0   # total time per trajectory
N_traj_bi = 50

rng_bi = np.random.default_rng(42)
X_bi_list, Y_bi_list, Z_bi_list = [], [], []

for _ in range(N_traj_bi):
    x0 = rng_bi.uniform(-1.2, 1.2, size=4)
    t_eval = np.arange(0, T_bistable + dt_bistable / 2, dt_bistable)
    sol = solve_ivp(bistable_system, [0, T_bistable], x0,
                    t_eval=t_eval, max_step=0.02, rtol=1e-8, atol=1e-10)
    if not sol.success:
        continue
    traj = sol.y.T  # [T_steps, 4]
    for k in range(len(traj) - 1):
        X_bi_list.append(traj[k])
        Y_bi_list.append(traj[k + 1])
        Z_bi_list.append(bistable_system.sigma(traj[k]))

X_bi = np.array(X_bi_list)
Y_bi = np.array(Y_bi_list)
Z_bi = np.array(Z_bi_list)

# Train/test split
perm_bi = rng_bi.permutation(len(X_bi))
n_tr_bi = int(0.8 * len(X_bi))
X_bi_tr, X_bi_te = X_bi[perm_bi[:n_tr_bi]], X_bi[perm_bi[n_tr_bi:]]
Y_bi_tr, Y_bi_te = Y_bi[perm_bi[:n_tr_bi]], Y_bi[perm_bi[n_tr_bi:]]
Z_bi_tr, Z_bi_te = Z_bi[perm_bi[:n_tr_bi]], Z_bi[perm_bi[n_tr_bi:]]

bi_train_ds = TensorDataset(torch.tensor(X_bi_tr, dtype=torch.float32),
                            torch.tensor(Y_bi_tr, dtype=torch.float32),
                            torch.tensor(Z_bi_tr, dtype=torch.long))
bi_test_ds = TensorDataset(torch.tensor(X_bi_te, dtype=torch.float32),
                           torch.tensor(Y_bi_te, dtype=torch.float32),
                           torch.tensor(Z_bi_te, dtype=torch.long))
bi_train_loader = DataLoader(bi_train_ds, batch_size=256, shuffle=True)
bi_test_loader = DataLoader(bi_test_ds, batch_size=512, shuffle=False)

print(f"Bistable data: {len(X_bi)} pairs ({n_tr_bi} train / {len(X_bi)-n_tr_bi} test)")
print(f"Mode distribution: {np.bincount(Z_bi, minlength=4)}")
print(f"State range: [{X_bi.min():.3f}, {X_bi.max():.3f}]")

In [ ]:
# ============================================================
# 10.3 Train physics-informed bank for bistable system
# ============================================================

def bistable_batch(X, mu):
    """Batched 1D bistable: mu*x - x^3. X: [B,N], mu: [N] or [B,N]."""
    if mu.dim() == 1:
        mu = mu.unsqueeze(0).expand(X.shape[0], -1)
    return mu * X - X ** 3

class BistableModeBank(nn.Module):
    def __init__(self, num_modes=4, num_nodes=4, mu_min=0.1, mu_max=1.5):
        super().__init__()
        self.num_modes = num_modes
        self.mu_min = mu_min; self.mu_max = mu_max
        self.graph_logits = nn.Parameter(torch.zeros(num_modes, num_nodes, num_nodes))
        self.mu_logits = nn.Parameter(torch.zeros(num_modes, num_nodes))
        self.alpha_param = nn.Parameter(torch.zeros(num_modes))
        self.beta_param = nn.Parameter(torch.zeros(num_modes))

    def graphs(self):
        S = 0.5 * (self.graph_logits + self.graph_logits.transpose(1, 2))
        W = torch.sigmoid(S)
        return W * (1.0 - torch.eye(W.shape[-1], device=W.device).unsqueeze(0))

    def mus(self):
        return self.mu_min + (self.mu_max - self.mu_min) * torch.sigmoid(self.mu_logits)

    def alphas(self): return self.alpha_param
    def betas(self): return self.beta_param

class BistableBank(nn.Module):
    def __init__(self, num_modes=4):
        super().__init__()
        self.bank = BistableModeBank(num_modes=num_modes)

    def mode_step(self, X, mode_idx, dt_val):
        """X: [B, N] -> [B, N] via Euler step."""
        W = self.bank.graphs()[mode_idx]
        mu = self.bank.mus()[mode_idx]
        a = self.bank.alphas()[mode_idx]
        b = self.bank.betas()[mode_idx]
        local = bistable_batch(X, mu)
        deg = W.sum(dim=-1)
        coupling = a * deg.unsqueeze(0) * X + b * torch.einsum("ij,bj->bi", W, X)
        return X + dt_val * (local + coupling)

    def forward_grouped(self, X, z, dt_val):
        out = torch.zeros_like(X)
        for r in range(self.bank.num_modes):
            idx = (z == r).nonzero(as_tuple=False).squeeze(-1)
            if idx.numel() > 0:
                out[idx] = self.mode_step(X[idx], r, dt_val)
        return out

# ---- Initialize with perturbed true parameters ----
torch.manual_seed(42)
rng_init = np.random.default_rng(7)
bi_bank = BistableBank(num_modes=4).to(device)

_mu_min, _mu_max = 0.1, 1.5
with torch.no_grad():
    for r, m in enumerate(bistable_mode_params):
        A_noisy = np.clip(m["A"] + rng_init.uniform(-0.15, 0.15, m["A"].shape), 1e-3, 1 - 1e-3)
        bi_bank.bank.graph_logits.data[r] = torch.tensor(
            np.log(A_noisy / (1 - A_noisy)), dtype=torch.float32)
        mu_sc = np.clip((m["mu"] + rng_init.uniform(-0.15, 0.15, 4) - _mu_min) / (_mu_max - _mu_min),
                        1e-3, 1 - 1e-3)
        bi_bank.bank.mu_logits.data[r] = torch.tensor(
            np.log(mu_sc / (1 - mu_sc)), dtype=torch.float32)
        bi_bank.bank.alpha_param.data[r] = m["alpha"] + rng_init.uniform(-0.02, 0.02)
        bi_bank.bank.beta_param.data[r] = m["beta"] + rng_init.uniform(-0.02, 0.02)

# ---- Train ----
opt_bi = optim.Adam(bi_bank.parameters(), lr=2e-3, weight_decay=1e-6)
bi_losses_tr, bi_losses_te = [], []

for epoch in range(1, 41):
    bi_bank.train()
    run_loss = 0.0; seen = 0
    for xb, yb, zb in bi_train_loader:
        xb, yb, zb = xb.to(device), yb.to(device), zb.to(device)
        pred = bi_bank.forward_grouped(xb, zb, dt_bistable)
        pred_loss = ((pred - yb) ** 2).mean()
        W = bi_bank.bank.graphs()
        loss = pred_loss + 1e-3 * (W * (1 - W)).mean()
        opt_bi.zero_grad(); loss.backward(); opt_bi.step()
        run_loss += pred_loss.item() * xb.shape[0]; seen += xb.shape[0]
    bi_losses_tr.append(run_loss / seen)

    bi_bank.eval()
    te_loss = 0.0; te_seen = 0
    with torch.no_grad():
        for xb, yb, zb in bi_test_loader:
            xb, yb, zb = xb.to(device), yb.to(device), zb.to(device)
            pred = bi_bank.forward_grouped(xb, zb, dt_bistable)
            te_loss += ((pred - yb) ** 2).mean().item() * xb.shape[0]; te_seen += xb.shape[0]
    bi_losses_te.append(te_loss / te_seen)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(bi_losses_tr, label="train"); ax.plot(bi_losses_te, label="test")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE")
ax.set_title("Bistable bank training"); ax.legend()
plt.tight_layout(); plt.show()

# ---- Print learned vs true parameters ----
W_learned = bi_bank.bank.graphs().detach().cpu().numpy()
mu_learned = bi_bank.bank.mus().detach().cpu().numpy()
a_learned = bi_bank.bank.alphas().detach().cpu().numpy()
b_learned = bi_bank.bank.betas().detach().cpu().numpy()

print("Learned vs true parameters:")
for r in range(4):
    m = bistable_mode_params[r]
    print(f"  Mode {r}: alpha={a_learned[r]:.4f} ({m['alpha']:.4f}), "
          f"beta={b_learned[r]:.4f} ({m['beta']:.4f}), "
          f"mu={np.round(mu_learned[r], 3)} ({m['mu']})")
    W_bin = (W_learned[r] > 0.5).astype(int)
    match = np.array_equal(W_bin, m["A"].astype(int))
    print(f"         graph binary match: {match}")

In [ ]:
# ============================================================
# 10.4 Morse graph analysis -- ground truth
# ============================================================

# Grid bounds from data range + margin
_data_min = X_bi.min(axis=0)
_data_max = X_bi.max(axis=0)
_margin = 0.3 * (_data_max - _data_min)
_bound = max(np.abs(_data_min - _margin).max(), np.abs(_data_max + _margin).max())
_bound = np.ceil(_bound * 2) / 2  # round to nearest 0.5

lb_morse = np.full(4, -_bound)
ub_morse = np.full(4,  _bound)
div_morse = np.full(4, 8, dtype=int)

grid_4d = UniformGrid(bounds=np.array([lb_morse, ub_morse]), divisions=div_morse)
n_boxes = int(np.prod(div_morse))

print(f"Grid: [{lb_morse[0]}, {ub_morse[0]}]^4, {div_morse[0]}^4 = {n_boxes} boxes")
print(f"Box size: {grid_4d.box_size}, diagonal: {np.linalg.norm(grid_4d.box_size):.4f}")
print(f"ODE integrations: {n_boxes * (2**4 + 1)}")

tau_morse = 1.0
F_gt = F_integration(bistable_system, tau_morse, epsilon=0.0)

print(f"\nComputing ground-truth Morse graph (tau={tau_morse})...")
t0 = time.time()
box_map_gt, morse_graph_gt, basins_gt = full_morse_graph_analysis(grid_4d, F_gt)
t_gt = time.time() - t0

print(f"Done in {t_gt:.1f}s")
print(f"  Morse sets: {len(morse_graph_gt.nodes())}")
print(f"  Edges:      {len(morse_graph_gt.edges())}")
for i, ms in enumerate(morse_graph_gt.nodes()):
    b = len(basins_gt.get(ms, set()))
    print(f"  Set {i}: {len(ms)} boxes, basin {b} boxes")

In [ ]:
# ============================================================
# 10.5 Morse graph analysis -- learned system
# ============================================================

# Build learned vector fields from trained bank parameters
learned_vfs = []
for r in range(4):
    W_bin = (W_learned[r] > 0.5).astype(float)  # threshold to binary
    learned_vfs.append(make_bistable_vf(W_bin, mu_learned[r], float(a_learned[r]), float(b_learned[r])))

learned_system = SwitchingSystem(bistable_polynomials, learned_vfs)

F_learned = F_integration(learned_system, tau_morse, epsilon=0.0)

print("Computing learned Morse graph...")
t0 = time.time()
box_map_l, morse_graph_l, basins_l = full_morse_graph_analysis(grid_4d, F_learned)
t_l = time.time() - t0

print(f"Done in {t_l:.1f}s")
print(f"  Morse sets: {len(morse_graph_l.nodes())}")
print(f"  Edges:      {len(morse_graph_l.edges())}")
for i, ms in enumerate(morse_graph_l.nodes()):
    b = len(basins_l.get(ms, set()))
    print(f"  Set {i}: {len(ms)} boxes, basin {b} boxes")

In [ ]:
# ============================================================
# 10.6 Comparison and visualization
# ============================================================

# ---- Summary table ----
print("=" * 65)
print("MORSE GRAPH COMPARISON: Ground Truth vs Learned")
print("=" * 65)
print(f"{'Metric':<30} {'GT':<15} {'Learned':<15}")
print("-" * 65)
print(f"{'Morse sets':<30} {len(morse_graph_gt.nodes()):<15} {len(morse_graph_l.nodes()):<15}")
print(f"{'Edges':<30} {len(morse_graph_gt.edges()):<15} {len(morse_graph_l.edges()):<15}")
total_basin_gt = sum(len(basins_gt.get(ms, set())) for ms in morse_graph_gt.nodes())
total_basin_l  = sum(len(basins_l.get(ms, set())) for ms in morse_graph_l.nodes())
print(f"{'Total basin boxes':<30} {total_basin_gt:<15} {total_basin_l:<15}")
print(f"{'Computation time (s)':<30} {t_gt:<15.1f} {t_l:<15.1f}")
print("=" * 65)

# ---- 2D projections of Morse sets and basins ----
all_boxes_4d = grid_4d.get_boxes()  # [N, 2, 4]
proj_pairs = [(0, 1, '$x_1$', '$x_2$'), (2, 3, '$x_3$', '$x_4$')]

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for col, (label, mg, bas) in enumerate([("Ground truth", morse_graph_gt, basins_gt),
                                         ("Learned", morse_graph_l, basins_l)]):
    n_ms = max(len(list(mg.nodes())), 1)
    cmap_colors = plt.cm.Set1(np.linspace(0, 0.9, n_ms))
    for row, (d1, d2, xl, yl) in enumerate(proj_pairs):
        ax = axes[row, col]
        # Basin boxes (light)
        for i, ms in enumerate(mg.nodes()):
            basin_idxs = list(bas.get(ms, set()))
            if basin_idxs:
                bboxes = all_boxes_4d[basin_idxs]
                bc = (bboxes[:, 0, :] + bboxes[:, 1, :]) / 2
                ax.scatter(bc[:, d1], bc[:, d2], s=2, c=[cmap_colors[i]], alpha=0.15)
        # Morse set boxes (solid)
        for i, ms in enumerate(mg.nodes()):
            ms_idxs = list(ms)
            if ms_idxs:
                mboxes = all_boxes_4d[ms_idxs]
                mc = (mboxes[:, 0, :] + mboxes[:, 1, :]) / 2
                ax.scatter(mc[:, d1], mc[:, d2], s=12, c=[cmap_colors[i]],
                          edgecolors='k', linewidths=0.3, label=f'MS {i}')
        # Switching surface
        x_line = np.linspace(lb_morse[d1], ub_morse[d1], 50)
        if d1 == 0:
            ax.plot(x_line, c1_bi - x_line, 'k--', alpha=0.4, linewidth=0.8)
        else:
            ax.plot(x_line, c2_bi - x_line, 'k--', alpha=0.4, linewidth=0.8)
        ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.set_title(f'{label}: {xl} vs {yl}')
        if row == 0:
            ax.legend(fontsize=7, markerscale=1.5, loc='best')

plt.suptitle('Morse sets and basins (2D projections)', y=1.01)
plt.tight_layout()
plt.show()

# ---- Mode partitioning ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax_idx, (d1, d2, xl, yl) in enumerate(proj_pairs):
    ax = axes[ax_idx]
    centers = (all_boxes_4d[:, 0, :] + all_boxes_4d[:, 1, :]) / 2
    box_modes = np.array([bistable_system.sigma(c) for c in centers])
    sc = ax.scatter(centers[:, d1], centers[:, d2], s=2, c=box_modes,
                   cmap='tab10', alpha=0.5, vmin=0, vmax=3)
    x_line = np.linspace(lb_morse[d1], ub_morse[d1], 50)
    if d1 == 0:
        ax.plot(x_line, c1_bi - x_line, 'k-', linewidth=1.5, label='$p_1=0$')
    else:
        ax.plot(x_line, c2_bi - x_line, 'k-', linewidth=1.5, label='$p_2=0$')
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.legend()
plt.colorbar(sc, ax=axes[-1], label='mode')
plt.suptitle('State-dependent mode partitioning')
plt.tight_layout()
plt.show()